# Notebook 02: Particle Transport Validation

This notebook validates the shielding transport module (`gcr/transport.py`):
- Stopping power curves vs NIST tabulated data
- CSDA range-energy relations
- Flux spectra before and after shielding
- Secondary particle production (protons + neutrons)
- Transport convergence with energy grid resolution

**Key references:**
- NIST PSTAR database — proton stopping powers
- Bradt & Peters (1950) — nuclear cross sections
- Zeitlin et al. (2013), Science 340, 1080 — MSL RAD transit measurement
- Slaba et al. (2014), Space Weather 12, 217 — HZETRN benchmark

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from gcr.transport import (
    load_stopping_power, bethe_bloch, proton_range,
    energy_after_slab, transport_flux_through_slab,
    bradt_peters_cross_section, nuclear_mean_free_path,
)
from gcr.spectrum import gcr_total_flux, load_usoskin_phi
from gcr.dose import let_from_energy, quality_factor_icrp60
from gcr.utils import ION_SPECIES, MATERIALS

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

E_grid = np.logspace(1, 5, 300)  # 10 MeV to 100 GeV
print('Transport validation notebook ready.')

## 1. Stopping Power — Bethe-Bloch vs NIST

Stopping power S(E) is the energy loss per unit areal density (MeV·cm²/g).
The pipeline uses Bethe-Bloch with NIST tabulated data for protons.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

materials = ['water', 'aluminum', 'polyethylene']
mat_colors = {'water': '#1f77b4', 'aluminum': '#ff7f0e', 'polyethylene': '#2ca02c'}
E_plot = np.logspace(0, 5, 300)  # 1 MeV to 100 GeV

# NIST reference values at key energies (proton, from PSTAR database)
# water: [10 MeV, 100 MeV, 1000 MeV] → [4.58, 0.772, 0.222] MeV·cm²/g
# aluminum: similar shape, different I value
nist_water_ref = {10.0: 4.58, 100.0: 0.772, 1000.0: 0.222}  # MeV·cm²/g

for i, mat in enumerate(materials):
    ax = axes[i]
    S_func = load_stopping_power(mat)
    S_vals = np.array([float(np.atleast_1d(S_func(e))[0]) for e in E_plot])
    
    ax.loglog(E_plot, S_vals, color=mat_colors[mat], linewidth=2,
              label=f'Pipeline (Bethe-Bloch)')
    
    if mat == 'water':
        ax.scatter(list(nist_water_ref.keys()), list(nist_water_ref.values()),
                   s=80, color='red', zorder=5, label='NIST PSTAR', marker='D')
    
    ax.set_xlabel('Proton kinetic energy (MeV)')
    ax.set_ylabel('Mass stopping power (MeV·cm²/g)')
    ax.set_title(f'{mat.title()}')
    ax.legend(fontsize=9)
    ax.set_xlim(1, 1e5)
    ax.set_ylim(0.05, 100)

plt.suptitle('Proton Stopping Power S(E) — Bethe-Bloch Parameterization',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/02_stopping_power.png', dpi=150, bbox_inches='tight')
plt.show()

# Validate against NIST reference values
print('\nValidation vs NIST PSTAR (proton in water):')
S_water = load_stopping_power('water')
for E_ref, S_ref in nist_water_ref.items():
    S_calc = float(np.atleast_1d(S_water(E_ref))[0])
    err = (S_calc - S_ref) / S_ref * 100
    print(f'  {E_ref:>6.0f} MeV: pipeline={S_calc:.3f}, NIST={S_ref:.3f} MeV·cm²/g  ({err:+.1f}%)')

## 2. HZE Ion Stopping Powers — Barkas-Bethe Scaling

Heavy ions have stopping power S_ion(E/A) = Z² × S_proton(E/A) at the same velocity.
Fe (Z=26) has S_Fe = 676 × S_proton — enormous energy loss and LET.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

species_colors = {'H': '#1f77b4', 'He': '#ff7f0e', 'C': '#2ca02c',
                  'O': '#d62728', 'Si': '#9467bd', 'Fe': '#8c564b'}
E_per_n = np.logspace(1, 5, 200)  # MeV/nucleon

S_water = load_stopping_power('water')
S_H = np.array([float(np.atleast_1d(S_water(e))[0]) for e in E_per_n])

ax.loglog(E_per_n, S_H, color=species_colors['H'], linewidth=2, label='H (Z=1)')

for sp_key, sp in ION_SPECIES.items():
    if sp_key == 'H':
        continue
    Z = sp['Z']
    S_ion = bethe_bloch(E_per_n, Z, 'water')
    ax.loglog(E_per_n, S_ion, color=species_colors[sp_key], linewidth=2,
              label=f"{sp_key} (Z={Z}, Z²={Z**2})")

ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel('Mass stopping power (MeV·cm²/g)')
ax.set_title('Stopping Power S(E/A) by Ion Species in Water\n'
             'S_ion(E/A) = Z² × S_proton(E/A) — Barkas-Bethe scaling')
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(10, 1e5)

plt.tight_layout()
plt.savefig('../figures/02_hze_stopping.png', dpi=150, bbox_inches='tight')
plt.show()

# Z² factors
print('\nZ² stopping power scaling factors (relative to proton):')
for sp_key, sp in ION_SPECIES.items():
    Z = sp['Z']
    print(f'  {sp_key:<4} (Z={Z:>2}): S = {Z**2:>4}× proton stopping power')

## 3. CSDA Range-Energy Relation

The CSDA range R(E) = ∫₀^E dE'/S(E') determines how far a particle travels before stopping.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: range in water vs energy for protons
ax = axes[0]
E_range = np.logspace(1, 4, 100)  # 10 MeV to 10 GeV
for mat, color in mat_colors.items():
    R = proton_range(E_range, mat)
    ax.loglog(E_range, R, color=color, linewidth=2, label=mat.title())

# NIST reference for water: 200 MeV proton → 25.9 g/cm²
ax.scatter([200], [25.9], s=100, color='red', zorder=5, marker='*',
           label='NIST: 200 MeV → 25.9 g/cm²')

ax.set_xlabel('Proton kinetic energy (MeV)')
ax.set_ylabel('CSDA range (g/cm²)')
ax.set_title('Proton CSDA Range in Shielding Materials')
ax.legend(fontsize=9)

# Right: energy after traversing different slab thicknesses (aluminum)
ax = axes[1]
E_in = np.logspace(1.5, 5, 100)  # 30 MeV to 100 GeV
thicknesses = [0, 5, 10, 20, 50]  # g/cm²
cmap = plt.cm.viridis
colors_thick = cmap(np.linspace(0, 1, len(thicknesses)))

for x, color in zip(thicknesses, colors_thick):
    E_out = np.array([energy_after_slab(e, x, 'aluminum') for e in E_in])
    mask = E_out > 0
    if np.any(mask):
        ax.loglog(E_in[mask], E_out[mask], color=color, linewidth=2,
                  label=f'{x} g/cm²')

ax.loglog(E_in, E_in, 'k--', linewidth=1, alpha=0.5, label='No shielding')
ax.set_xlabel('Incident energy (MeV/nucleon)')
ax.set_ylabel('Exit energy (MeV/nucleon)')
ax.set_title('Energy Loss Through Aluminum Shielding (Protons)')
ax.legend(fontsize=9, title='Thickness')

plt.suptitle('CSDA Range and Energy Loss', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/02_range_energy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'NIST validation: proton range at 200 MeV in water = '
      f"{proton_range(np.array([200.0]), 'water')[0]:.1f} g/cm² (NIST: 25.9 g/cm²)")

## 4. GCR Flux Spectrum — Before and After Shielding

Shielding removes low-energy particles (they stop) but high-energy GCR passes through.
The transported spectrum is harder (higher mean energy) than the incident spectrum.

In [ ]:
phi_msl = 550.0  # MV (MSL cruise conditions)
flux_in = gcr_total_flux(E_grid, phi_msl)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: proton flux before/after different aluminum thicknesses
ax = axes[0]
ax.loglog(E_grid, flux_in['H'], 'k-', linewidth=2, label='Incident (0 g/cm²)')

thicknesses_plot = [5, 10, 16, 30]
colors_plot = plt.cm.Reds(np.linspace(0.4, 1.0, len(thicknesses_plot)))

for x, color in zip(thicknesses_plot, colors_plot):
    flux_transported = transport_flux_through_slab(flux_in, x, 'aluminum', E_grid)
    ax.loglog(E_grid, flux_transported['H'], color=color, linewidth=1.8,
              label=f'{x} g/cm² Al')

ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel(r'Proton flux (cm$^{-2}$ s$^{-1}$ MeV$^{-1}$ sr$^{-1}$)')
ax.set_title('Proton Flux: Incident vs Transported (Aluminum)')
ax.legend(fontsize=9)
ax.set_xlim(10, 1e5)
ax.axvline(938, color='gray', linestyle=':', alpha=0.4, label='938 MeV = mp·c²')

# Right: all species at 16 g/cm² Al (MSL shielding equivalent)
ax = axes[1]
x_msl = 16.0
flux_msl = transport_flux_through_slab(flux_in, x_msl, 'aluminum', E_grid)

for sp_key in ['H', 'He', 'C', 'O', 'Si', 'Fe']:
    sp = ION_SPECIES[sp_key]
    ax.loglog(E_grid, flux_in[sp_key], color=species_colors[sp_key],
              linewidth=1.5, linestyle='--', alpha=0.5)
    ax.loglog(E_grid, flux_msl[sp_key], color=species_colors[sp_key],
              linewidth=2, label=sp_key)

# Show neutrons (secondary)
if 'neutron' in flux_msl:
    ax.loglog(E_grid, flux_msl['neutron'], 'k:', linewidth=1.5,
              alpha=0.7, label='Neutron (secondary)')

ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel(r'Differential flux (cm$^{-2}$ s$^{-1}$ MeV$^{-1}$ sr$^{-1}$)')
ax.set_title(f'All Species: Incident (dashed) vs {x_msl:.0f} g/cm² Al (solid)')
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(10, 1e5)

plt.suptitle(f'GCR Transport Through Aluminum Shielding (φ = {phi_msl:.0f} MV)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/02_transported_flux.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. LET Distribution and Quality Factor Q(L)

The quality factor Q(L) converts absorbed dose to dose equivalent, weighting by biological effectiveness. High-LET HZE ions have Q up to ~30 (Fe at Bragg peak).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Q(L) curve with species LET bands
ax = axes[0]
L_range = np.logspace(-1, 4, 500)  # keV/um
Q_range = quality_factor_icrp60(L_range)

ax.loglog(L_range, Q_range, 'k-', linewidth=2.5, label='ICRP-60 Q(L)')

# Shade regions
ax.fill_between(L_range[L_range < 10], 0.5, Q_range[L_range < 10],
                alpha=0.15, color='blue', label='Q=1 region (L<10 keV/μm)')
ax.fill_between(L_range[(L_range >= 10) & (L_range <= 100)], 0.5,
                Q_range[(L_range >= 10) & (L_range <= 100)],
                alpha=0.15, color='orange', label='Rising Q region')
ax.fill_between(L_range[L_range > 100], 0.5, Q_range[L_range > 100],
                alpha=0.15, color='red', label='300/√L region')

# Mark typical LET at 1 GeV/n for each species
E_ref = np.array([1000.0])  # 1 GeV/n
for sp_key, sp in ION_SPECIES.items():
    LET = let_from_energy(E_ref, sp['Z'])[0]
    Q = quality_factor_icrp60(np.array([LET]))[0]
    ax.scatter([LET], [Q], s=80, color=species_colors[sp_key], zorder=5)
    ax.annotate(f' {sp_key}', (LET, Q), fontsize=8, va='center')

ax.set_xlabel('LET (keV/μm in tissue)')
ax.set_ylabel('Quality Factor Q(L)')
ax.set_title('ICRP-60 Quality Factor Q(L)\npoints = species at 1 GeV/n')
ax.legend(fontsize=8)
ax.set_ylim(0.5, 50)
ax.set_xlim(0.1, 1e4)

# Right: LET vs energy for all species
ax = axes[1]
E_let = np.logspace(1, 5, 200)
for sp_key, sp in ION_SPECIES.items():
    LET = let_from_energy(E_let, sp['Z'])
    ax.loglog(E_let, LET, color=species_colors[sp_key], linewidth=2,
              label=f"{sp_key} (Z={sp['Z']})")

ax.axhline(10, color='orange', linestyle='--', alpha=0.7, label='Q=1 threshold (L=10)')
ax.axhline(100, color='red', linestyle='--', alpha=0.7, label='Q max threshold (L=100)')
ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel('LET (keV/μm in tissue)')
ax.set_title('LET vs Energy by Ion Species')
ax.legend(fontsize=9)
ax.set_xlim(10, 1e5)

plt.suptitle('LET Distribution and ICRP-60 Quality Factor', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/02_let_quality_factor.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Nuclear Cross Sections and Mean Free Path

Nuclear interactions attenuate the primary GCR beam and produce secondaries.
The nuclear mean free path λ = A/(NA × σ_R × ρ) determines survival probability exp(-x/λ).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: nuclear cross sections vs ion species in aluminum
ax = axes[0]
species_list = list(ION_SPECIES.items())
sp_names = [f"{k} (Z={v['Z']})" for k, v in species_list]
sigmas_barn = [bradt_peters_cross_section(v['Z'], v['A'], 13, 27) / 1e-24
               for k, v in species_list]
mfps_al = [nuclear_mean_free_path(v['Z'], v['A'], 'aluminum')
           for k, v in species_list]

colors_bar = [species_colors[k] for k, v in species_list]
bars = ax.bar(range(len(sp_names)), sigmas_barn, color=colors_bar, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(sp_names)))
ax.set_xticklabels(sp_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Nuclear reaction cross section (barn)')
ax.set_title('Bradt-Peters Cross Sections in Aluminum')

# Right: mean free path and survival probability at 16 g/cm²
ax = axes[1]
survival_16 = [np.exp(-16.0 / mfp) * 100 for mfp in mfps_al]

bars2 = ax.bar(range(len(sp_names)), survival_16, color=colors_bar,
               edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(sp_names)))
ax.set_xticklabels(sp_names, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Nuclear survival fraction (%)')
ax.set_title('Nuclear Survival at 16 g/cm² Aluminum')

for i, (mfp, surv) in enumerate(zip(mfps_al, survival_16)):
    ax.text(i, surv + 1, f'λ={mfp:.0f}\ng/cm²', ha='center', fontsize=7)

plt.suptitle('Nuclear Cross Sections and Mean Free Paths', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/02_nuclear_mfp.png', dpi=150, bbox_inches='tight')
plt.show()

print('Nuclear mean free paths in aluminum (g/cm²):')
for (k, v), mfp, surv in zip(species_list, mfps_al, survival_16):
    print(f'  {k:<4}: λ = {mfp:6.1f} g/cm²,  survival at 16 g/cm² = {surv:.1f}%')

## 7. Dose Rate vs Shielding Thickness — comparison with HZETRN

Comparing pipeline output against published HZETRN predictions for MSL cruise conditions.

**References:**
- Slaba et al. (2014), Space Weather 12, 217: HZETRN predictions at MSL conditions
- Mrigakshi et al. (2013), JGR Space Phys 118, 6633: HZETRN dose rate range

In [ ]:
from gcr.dose import dose_rate_from_flux, dose_equivalent_rate

phi_msl = 550.0  # MV
thicknesses = np.array([0, 5, 10, 16, 20, 30, 50])  # g/cm²
flux_in_msl = gcr_total_flux(E_grid, phi_msl)

D_rates = []
H_rates = []

for x in thicknesses:
    if x > 0:
        flux_t = transport_flux_through_slab(flux_in_msl, x, 'aluminum', E_grid)
    else:
        flux_t = flux_in_msl
    
    dr = dose_rate_from_flux(flux_t, E_grid)
    hr = dose_equivalent_rate(flux_t, E_grid)
    D_rates.append(dr['dose_rate_mGy_day'])
    H_rates.append(hr['H_rate_mSv_day'])

D_rates = np.array(D_rates)
H_rates = np.array(H_rates)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: dose rate vs shielding
ax = axes[0]
ax.plot(thicknesses, D_rates, 'b-o', linewidth=2, markersize=7,
        label='Pipeline (absorbed dose)')
ax.plot(thicknesses, H_rates, 'r-s', linewidth=2, markersize=7,
        label='Pipeline (dose equivalent)')

# HZETRN reference range at 16 g/cm² (Mrigakshi 2013, Slaba 2014)
ax.axvspan(14, 18, alpha=0.1, color='blue')
ax.errorbar([16], [1.84], yerr=[[0.14], [0.14]], fmt='k*', markersize=15,
            capsize=8, linewidth=2, label='RAD measured (Zeitlin 2013)')
ax.fill_between([0, 60], [1.70, 1.70], [2.10, 2.10], alpha=0.1, color='green',
                label='HZETRN envelope (Mrigakshi 2013)')
ax.axhline(1.75, color='green', linestyle='--', linewidth=1.5, alpha=0.7,
           label='HZETRN best estimate (Slaba 2014)')

ax.set_xlabel('Aluminum shielding (g/cm²)')
ax.set_ylabel('Dose rate (mGy/day or mSv/day)')
ax.set_title('Dose Rate vs Shielding Thickness')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(-2, 55)

# Right: Q_effective vs shielding
ax = axes[1]
Q_eff = H_rates / np.maximum(D_rates, 1e-10)
ax.plot(thicknesses, Q_eff, 'k-^', linewidth=2, markersize=7,
        label='Pipeline Q_eff')
ax.axhline(2.62, color='orange', linestyle='--', linewidth=2,
           label='RAD measured Q_eff=2.62 (Zeitlin 2013)')

ax.set_xlabel('Aluminum shielding (g/cm²)')
ax.set_ylabel('Effective quality factor Q_eff = H/D')
ax.set_title('Q_eff vs Shielding Thickness')
ax.legend(fontsize=9)
ax.set_xlim(-2, 55)
ax.set_ylim(1, 8)

plt.suptitle('Pipeline Validation vs HZETRN and MSL RAD (φ=550 MV, MSL conditions)',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('../figures/02_dose_vs_shielding.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDose rate vs shielding (aluminum, MSL conditions):')
print(f'{"Thickness":<12} {"D (mGy/d)":<12} {"H (mSv/d)":<12} {"Q_eff"}')
print('-' * 48)
for x, D, H, Q in zip(thicknesses, D_rates, H_rates, Q_eff):
    print(f'{x:<12.0f} {D:<12.2f} {H:<12.2f} {Q:.2f}')
print('\nReference (Zeitlin 2013): D=1.84 mGy/d, H=4.81 mSv/d, Q_eff=2.62')
print('Reference (HZETRN/Mrigakshi): D=1.70–2.10 mGy/d at 16 g/cm² Al')